In [1]:
# 座標法による多角形面積計算
import Pkg
c = ["DataFrames", "Metal", "Chain", "BenchmarkTools"]
Pkg.add(c)

using DataFrames, Metal, Chain, BenchmarkTools

# カーネル関数を使った多角形の面積計算（座標法）
function kernel_shoelace(isovist_partial::MtlDeviceArray{Float32, 2},
                            crossing_x::MtlDeviceArray{Float32, 2}, crossing_y::MtlDeviceArray{Float32, 2},
                            crossing_next_x::MtlDeviceArray{Float32, 2}, crossing_next_y::MtlDeviceArray{Float32, 2})::Nothing

    i = thread_position_in_grid_2d().x
    j = thread_position_in_grid_2d().y

    if i > size(isovist_partial, 1) || j > size(isovist_partial, 2)
        return nothing
    end

    isovist_partial[i, j] = (crossing_x[i, j] * crossing_next_y[i, j] - 
                                crossing_y[i, j] * crossing_next_x[i, j]) / 2.0f0

    return nothing

end

# カーネル関数により行列要素を1つ後ろにずらす
function kernel_shift_row_elements(crossing_next_x::MtlDeviceArray{Float32, 2}, crossing_next_y::MtlDeviceArray{Float32, 2},
                                    crossing_x::MtlDeviceArray{Float32, 2}, crossing_y::MtlDeviceArray{Float32, 2})::Nothing

    i = thread_position_in_grid_2d().x
    j = thread_position_in_grid_2d().y

    if i > size(crossing_x, 1) || j > size(crossing_x, 2)
        return nothing
    end

    crossing_next_x[i, j] = crossing_x[i, mod1(j + 1, size(crossing_x, 2))]
    crossing_next_y[i, j] = crossing_y[i, mod1(j + 1, size(crossing_y, 2))]

    return nothing

end

function main()
    crossing_x = [0.0 10.0 10.0 5.0 0.0;
                  0.0 10.0 10.0 5.0 0.0]
    crossing_y = [0.0 0.0 10.0 5.0 10.0;
                  0.0 0.0 10.0 20.0 10.0]

    Mtl_crossing_x = MtlArray{Float32}(crossing_x)
    Mtl_crossing_y = MtlArray{Float32}(crossing_y)
    Mtl_crossing_next_x = similar(Mtl_crossing_x)
    Mtl_crossing_next_y = similar(Mtl_crossing_y)

    num_points = size(crossing_x, 1)
    div_n = size(crossing_x, 2)

    threads_per_group = (32, 32)
    num_groups = (ceil(Int, num_points / threads_per_group[1]), ceil(Int, div_n / threads_per_group[2]))
    @metal threads = threads_per_group groups = num_groups kernel_shift_row_elements(Mtl_crossing_next_x, Mtl_crossing_next_y,
                                                                                    Mtl_crossing_x, Mtl_crossing_y)

    Mtl_isovist_partial = MtlArray{Float32}(undef, num_points, div_n)

    @metal threads = threads_per_group groups = num_groups kernel_shoelace(Mtl_isovist_partial, 
                                                                        Mtl_crossing_x, Mtl_crossing_y,
                                                                        Mtl_crossing_next_x, Mtl_crossing_next_y)

    isovist = @chain Mtl_isovist_partial begin
        sum(_, dims = 2)
        reshape(_, num_points, 1)
    end
    
end
@benchmark main()

   Resolving package versions...
   Installed Preferences ──────── v1.5.0
   Installed Adapt ────────────── v4.4.0
   Installed Parsers ──────────── v2.8.3
   Installed BenchmarkTools ───── v1.6.3
   Installed JSON ─────────────── v1.3.0
   Installed PrettyTables ─────── v3.1.1
   Installed JLLWrappers ──────── v1.7.1
   Installed InlineStrings ────── v1.4.5
   Installed Tables ───────────── v1.12.1
   Installed StaticArrays ─────── v1.9.15
   Installed StaticArraysCore ─── v1.4.4
   Installed OrderedCollections ─ v1.8.1
   Installed Scratch ──────────── v1.3.0
   Installed PrecompileTools ──── v1.3.3
   Installed GPUCompiler ──────── v1.7.4
   Installed ScopedValues ─────── v1.5.0
   Installed Random123 ────────── v1.7.1
   Installed Tracy ────────────── v0.1.6
   Installed DataFrames ───────── v1.8.1
   Installed RandomNumbers ────── v1.6.0
   Installed LLVMExtra_jll ────── v0.0.38+0
   Installed GPUToolbox ───────── v1.0.0
   Installed BFloat16s ────────── v0.6.0
   Installed GPUArr

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  330.041 μs …  3.584 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     396.667 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   409.005 μs ± 65.378 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

              ▂▄▆▇█▇▆▄▂▁                                        
  ▁▁▁▁▁▁▁▁▂▃▅▇██████████▇▆▅▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▂▂▂▂▂▂▁▂▁▁▁▁ ▃
  330 μs          Histogram: frequency by time          553 μs <

 Memory estimate: 14.14 KiB, allocs estimate: 494.

In [15]:
# 座標法による多角形面積計算
import Pkg
c = ["DataFrames", "Metal", "Chain", "BenchmarkTools"]
Pkg.add(c)

using DataFrames, Metal, Chain, BenchmarkTools

# カーネル関数を使った多角形の面積計算（座標法）
function kernel_shoelace(isovist_partial::MtlDeviceArray{Float32, 2},
                            crossing_x::MtlDeviceArray{Float32, 2}, crossing_y::MtlDeviceArray{Float32, 2})::Nothing

    i = thread_position_in_grid_2d().x
    j = thread_position_in_grid_2d().y

    if i > size(isovist_partial, 1) || j > size(isovist_partial, 2)
        return nothing
    end

    isovist_partial[i, j] = (crossing_x[i, j] * crossing_y[i, mod1(j + 1, size(crossing_y, 2))] - 
                                crossing_y[i, j] * crossing_x[i, mod1(j + 1, size(crossing_x, 2))]) / 2.0f0

    return nothing

end

function main()
    crossing_x = [0.0 10.0 10.0 5.0 0.0;
                  0.0 10.0 10.0 5.0 0.0]
    crossing_y = [0.0 0.0 10.0 5.0 10.0;
                  0.0 0.0 10.0 20.0 10.0]

    Mtl_crossing_x = MtlArray{Float32}(crossing_x)
    Mtl_crossing_y = MtlArray{Float32}(crossing_y)
    Mtl_isovist_partial = similar(Mtl_crossing_x)

    num_points = size(crossing_x, 1)
    div_n = size(crossing_x, 2)

    threads_per_group = (32, 32)
    num_groups = (ceil(Int, num_points / threads_per_group[1]), ceil(Int, div_n / threads_per_group[2]))

    @metal threads = threads_per_group groups = num_groups kernel_shoelace(Mtl_isovist_partial, 
                                                                            Mtl_crossing_x, Mtl_crossing_y)

    isovist = @chain Mtl_isovist_partial begin
        sum(_, dims = 2)
        reshape(_, num_points, 1)
    end
    
end

@benchmark main()

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


BenchmarkTools.Trial: 3861 samples with 1 evaluation per sample.
 Range (min … max):  488.125 μs … 41.510 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     566.250 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):     1.293 ms ±  4.608 ms  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █                                                             
  ██▆▄▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▃▁▃▁▁▁▃▁▁▁▃▁▄▃▅▃▄▆▅▆▆▅ █
  488 μs        Histogram: log(frequency) by time      31.7 ms <

 Memory estimate: 15.49 KiB, allocs estimate: 605.